### LIBRARY IMPORTS

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

### RIDGE REGRESSION

In [19]:
ridge = Ridge(alpha=1)
ridge.fit(X_train, y_train)

ridge_valid_preds = ridge.predict(X_valid)

print(f"Ridge validation MSE: {mean_squared_error(y_valid, ridge_valid_preds)}")
print(f"Ridge validation R^2: {r2_score(y_valid, ridge_valid_preds)}")

Ridge validation MSE: 0.10518635605186145
Ridge validation R^2: 0.7779548987729593


### DECISION TREE

In [21]:
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

dt_valid_preds = dt.predict(X_valid)

print(f"Ridge validation MSE: {mean_squared_error(y_valid, dt_valid_preds)}")
print(f"Ridge validation R^2: {r2_score(y_valid, dt_valid_preds)}")

Ridge validation MSE: 0.1549803498599775
Ridge validation R^2: 0.6728413383205902


### RANDOM FOREST

In [22]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_test_preds = rf.predict(X_valid)

print(f"Random forest validation MSE: {mean_squared_error(y_valid, rf_test_preds)}")
print(f"Random forest validation R^2: {r2_score(y_valid, rf_test_preds)}")

Random forest validation MSE: 0.15121166899490873
Random forest validation R^2: 0.6807969055213783


### NEURAL NETWORK

In [3]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        patience: int = 10
    ) -> None:
        
        input_size = X_train.shape[1]
        output_size = y_train.shape[1] if len(y_train.shape) > 1 else 1
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.float32).view(-1, output_size).to(self.device)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.view_as(preds))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                val_preds_np = val_preds.cpu().numpy().flatten()
                y_valid_np = y_valid_t.cpu().numpy().flatten()
                val_r2 = r2_score(y_valid_np, val_preds_np)

            print(f"Epoch: {epoch+1} | Validation MSE: {val_loss:.4f} | R^2: {val_r2:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [5]:
mlp = MLP(epochs=100, learning_rate=0.0001, dropout=0.3, hidden_size=[16, 8], batch_size=256)
mlp.fit(X_train, y_train, X_valid, y_valid)

Epoch: 1 | Validation MSE: 0.1289 | R^2: 0.7279
Epoch: 2 | Validation MSE: 0.1157 | R^2: 0.7557
Epoch: 3 | Validation MSE: 0.1126 | R^2: 0.7623
Epoch: 4 | Validation MSE: 0.1121 | R^2: 0.7633
Epoch: 5 | Validation MSE: 0.1122 | R^2: 0.7631
Epoch: 6 | Validation MSE: 0.1116 | R^2: 0.7645
Epoch: 7 | Validation MSE: 0.1109 | R^2: 0.7659


KeyboardInterrupt: 